In [ ]:
from google.colab import auth

# Authenticate user credentials for google colab
auth.authenticate_user()
print('Authenticated')

Authenticated


In [ ]:
from google.cloud import bigquery

# Declare the default bigquery project to use
project = 'ld-pcx-bia'

# Connect to bigquery
client = bigquery.Client(project=project)

In [ ]:
# GoogleSheet should be filled out with the SDM and PCX liams (https://docs.google.com/spreadsheets/d/1a1aqmAGn17F37dT-vJkT_HDP7DMzbCJV3SghhNTgiFU/edit?gid=0#gid=0)

# Create a dataframe called df with the saved results of the query run

df = client.query('''

  SELECT * FROM `ld-pcx-bia.jongrub.assetful_image_download_prep`

''' ).to_dataframe()

# Preview that the dataframe looks as expected
print(df.head())

            sdm_liam            sdm_filename_ext  \
0  SDM_4059729421609  4059729421609_enfr_01.jpeg   
1  SDM_4059729421609  4059729421609_enfr_05.jpeg   
2  SDM_4059729518729  4059729518729_enfr_05.jpeg   
3  SDM_4059729544131  4059729544131_enfr_03.jpeg   
4  SDM_3616305980908  3616305980908_enfr_03.jpeg   

                                        assetful_url        pcx_liam  \
0  https://digital.loblaws.ca/SDM/SDM_40597294216...  21673060004_EA   
1  https://digital.loblaws.ca/SDM/SDM_40597294216...  21673060004_EA   
2  https://digital.loblaws.ca/SDM/SDM_40597295187...  21676940002_EA   
3  https://digital.loblaws.ca/SDM/SDM_40597295441...  21672730001_EA   
4  https://digital.loblaws.ca/SDM/SDM_36163059809...  21672652001_EA   

   pcx_article           pcx_filename_ext  
0  21673060004  21673060004_fr_front.jpeg  
1  21673060004    21673060004_fr_top.jpeg  
2  21676940002    21676940002_fr_top.jpeg  
3  21672730001   21672730001_en_side.jpeg  
4  21672652001   21672652001_fr_si

In [ ]:
import os
from datetime import datetime

# Create a folder called "/zip_image_downloads" and make it the directory
os.makedirs('/zip_image_downloads', exist_ok=True)
os.chdir('/zip_image_downloads')
current_wd = os.getcwd()

# Create and use a directory for the current date
current_date = datetime.now().strftime("%Y_%m_%d")
current_date_dir = os.path.join(current_wd, current_date)
os.makedirs(current_date_dir, exist_ok=True)
os.chdir(current_date_dir)

print(f"Current working directory: {os.getcwd()}")

Current working directory: /zip_image_downloads/2025_07_03


In [1]:
import os
import requests
import pandas as pd
from datetime import datetime
import subprocess

# Download the dataframe as a csv with all liams and image urls
df.to_csv('output.csv')

def download_images(df: pd.DataFrame):
    """
    Downloads images from URLs in a DataFrame, saves them to a directory structure,
    and then creates a zip archive of the directory using a shell command.

    Args:
        df (pd.DataFrame): DataFrame with columns 'pcx_liam', 'sdm_liam', 'assetful_url', and 'pcx_filename_ext'.
    """

    for index, row in df.iterrows():
        liam = row['pcx_liam']
        url = row['assetful_url']
        file_name = row['pcx_filename_ext']

        # Download all image files and save them to their current date subfolder
        file_path = os.path.join(current_date_dir, file_name)
        try:
            response = requests.get(url, stream=True)
            response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)

            with open(file_path, 'wb') as file:
                for chunk in response.iter_content(chunk_size=8192):  # 8KB chunks
                    file.write(chunk)

            print(f"Downloaded {file_name} from {url} and saved to {file_path}")

        except requests.exceptions.RequestException as e:
            print(f"Error downloading {file_name} from {url}: {e}")
        except Exception as e:
            print(f"Error saving {file_name} to {file_path}: {e}")

    # Create a zip archive of the current_date_dir using a shell command
    zip_file_path = os.path.join(current_date_dir, f"{current_date}.zip")
    try:
        # Construct the zip command.  The -r flag is for recursive, and -q for quiet.
        command = f"zip -r -q {zip_file_path} {current_date_dir}"

        # Execute the command using subprocess.run
        subprocess.run(command, shell=True, check=True)  # shell=True is necessary to interpret the command string

        print(f"Created zip archive: {zip_file_path}")

    except subprocess.CalledProcessError as e:
        print(f"Error creating zip archive: {e}")

download_images(df)

NameError: name 'df' is not defined